# Use Ollama for Web Search

1) Install ollama on your laptop (https://ollama.com/download)
2) Create a free account (https://signin.ollama.com/)
3) Create a a free API key and copy it (https://ollama.com/settings/keys)
4) Adapt the script below to loop over a file of links.

Note: This likely needs to run locally (not on HPC) because of how Ollama works. If you don't have Python installed, the easiest way to do so is by downloading Anaconda (https://www.anaconda.com/download/success?reg=skipped)

In [8]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [9]:
#imports and client

import os
import pandas as pd
from ollama import Client

#retrieve API key
API_KEY = os.getenv("OLLAMA_API_KEY")

client = Client(
    host="http://localhost:11434",
    headers={"Authorization": f"Bearer {API_KEY}"}
)

tools = {
    "web_search":client.web_search,
    "web_fetch":client.web_fetch,
}

In [10]:
#read dataset

farms = pd.read_csv("filtered_dallas_farms.csv")

In [ ]:
#make api key
API_KEY = "bbde8f7d702d4546b322e4fb98235f4d.2a-C9SLJm-aB0_T8sjCsKpMD"

client = Client(
    host="http://localhost:11434",
    headers={"Authorization": f"Bearer {API_KEY}"}
)

In [12]:
#redefine tools 
tools = {
    "web_search": client.web_search,
    "web_fetch": client.web_fetch
}

In [ ]:
#test
print(API_KEY[:5])
print(API_KEY is None)

bbde8
False


In [ ]:
#webscraping function
def research_farm(farm_name, website):
    messages = [
    {
        "role": "user", 
        "content": f""" Research this urban farm:
        Farm name: {farm_name}
        Website from dataset: {website}

        Use the provided website first. Do not switch to a different farm with a similar name.
        Do not count a donation/cart page as produce sales unless the page clearly mentions produce, CSA, farm stand, market, restaurant sales, or food sales.
        If a website is provided, use web_fetch on that website first. Do not use web_search unless the website is missing or unusable.
        Do not say sells_produce = yes unless the website clearly mentions produce sales, CSA memberships, farm stands, farmers markets, restaurant sales, wholesale produce, or direct sales to customers.
        Do not count donation pages, membership pages, event pages, volunteer opportunities, educational programs, or a general shopping cart as evidence of produce sales.
        If there is not enough evidence, set sells_produce = no evidence found.
        
        Determine:
        1. Does it have a website?
        2. If not, can you find an official website or social media page?
        3. Does it sell edible produce through Community Supported Agriculture (CSA), a farm stand, farmers market, online store, restaurants, or direct sales?
        4. If it only sells flowers or honey, do not classify it as selling produce.
        5. Does it only donate produce, provide education, or run community programs?
        6. If there is no useful information, say "no evidence found."

        Return a short answer with:
    - website_found:
    - social_media_found:
    - sells_produce:
    - sells_only_flowers_or_honey:
    - evidence:
    - classification:
        """
        }
    ]

    while True: 
        response = client.chat(
            model="llama3.2:3b",
            messages=messages,
            tools=[client.web_search, client.web_fetch]
        )

        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content
        
        for call in response.message.tool_calls:
            tool_name = call.function.name
            args = call.function.arguments

            if tool_name == "web_search":
                result = client.web_search(**args)

            elif tool_name == "web_fetch":
                if "url" in args:
                    result = client.web_fetch(args["url"])
            elif "q" in args:
                result = client.web_search(args["q"])
            else:
                result = f"Bad arguments: {args}"

            messages.append({
                "role": "tool",
                "content": str(result)
        })
            

In [15]:
print(farms.columns.tolist())

['name', 'address', 'lat', 'lon', 'Website', 'Google Categories', 'Place ID', 'Status', 'profile_url', 'osm_id', 'fclass', 'county', 'COUNTYFP', 'house_number', 'street', 'city', 'zipcode', 'full_address', 'Unnamed: 0', 'latitude', 'longitude', 'oid', 'geoid', 'state', 'farm_id', 'farm_name', 'street_address', 'zip_code', 'latitude_1', 'date_collected', 'website', 'index_right', 'MTFCC', 'OID', 'GEOID', 'STATE', 'COUNTY', 'COUNTYNS', 'BASENAME', 'NAME', 'LSADC', 'FUNCSTAT', 'COUNTYCC', 'AREALAND', 'AREAWATER', 'OBJECTID', 'CENTLAT', 'CENTLON', 'INTPTLAT', 'INTPTLON']


In [16]:
#when one farm works

#farms["llm_result"] = farms.apply(
    #lambda row: research_farm(
        #row["Farm"],
        #row["Website"]
    #),
   # axis=1
#)

In [17]:
client.web_fetch("https://www.okofarms.org/")

WebFetchResponse(title='Oko Farms', content="Oko Farms\n\nDONATE\n\n### URBAN FARMING, EDUCATION, AND ENVIRONMENTAL STEWARDSHIP IN BROOKLYN,NY\n\nOko Farms (est. 2013) is New York City's only publicly accessible aquaponics farm, educational center, and community hub. Using aquaponics, we sustainably grow fish and plants together in a recirculating ecosystem to save water and grow more food in small, urban spaces.\n\nThe word “oko” pays homage to our founder’s Yoruba heritage. Oko is a Yoruba word which loosely translates to farm in English. A more accurate definition of the word is a province or place where agriculture is at the center of socio-economic life, daily activities, and cultural traditions.\n\n#### MISSION\n\n#### Oko Farms’ mission is to use aquaponics farming as a tool to increase food security, combat climate change and strengthen community resilience.\n\n#### WHAT WE GROW\n\nWe cultivate a wide variety of vegetables, herbs, fruits, medicinal plants and flowers that demon

In [23]:
website = farms.loc[0, "Website"]
print(website)

client.web_fetch(website)

https://www.elmwoodfarm.co/


WebFetchResponse(title='Elmwood Farm | Join Our Sustainable Community Effort', content="Elmwood Farm | Join Our Sustainable Community Effort\n\n### Cultivating healthy relationships between Land & Neighbor in Oak Cliff, Texas\n\nJoin Our Newsletter\n\nBecome a Member Today!\n\n## Help Elmwood Farm plant long-term roots in Oak Cliff.\n\n$10.00\n\n$20.00\n\n$30.00\n\n$40.00\n\nCustom Amount\n\nPlease enter an amount\n\n$\n\nOne-Time Donation Weekly Donation Monthly Donation\n\nDonate\n\n## Elmwood Farm is a one-acre urban farm where meaningful work, play, and rest all come together.\n\n## A deeper relationship with your food and your neighbors\n\n#### Urban Farming\n\nModeling a holistic approach for truly sustainable agriculture at any scale.\n\n#### Neighborhood Events\n\nDinners, concerts, workshops, and weekly playgroups foster an open community green space.\n\n#### Community Composting\n\nWith the help of our neighbors, we divert food waste while building soil fertility.\n\n# Don’t 

In [24]:
result = research_farm(
    farms.loc[0, "farm_name"],
    farms.loc[0, "Website"] if pd.notna(farms.loc[0, "Website"]) else ""
)

print(result)

The article discusses the farm foundations of Elmwood Farm, which includes two main aspects: animals and tree crops. The author explains that these elements pose challenges and risks in an urban context but are essential for sustainable farming.

*   Animals:
    *   Livestock provide a significant source of farm fertility and help with soil health.
    *   Grazing animals regenerate and maintain grasslands through their grazing, fertilizing, and soil disturbance activities.
*   Tree Crops:
    *   Fruit and nut trees serve as the food and financial anchor of the farm, providing consistent, low-maintenance crops for decades.
    *   Taller overstory trees provide necessary shade for crops, livestock, and humans in the summer, while smaller trees like persimmon and mulberry offer fruit for birds and pigs in the winter.

**Benefits of Membership**

*   Access to the farm 24/7
*   Discounts at events and online store
*   Online community for sharing resources and getting to know like-mind

In [25]:
print(API_KEY is None)
print(API_KEY[:8])

client.web_search("test urban farm")
client.web_fetch("https://example.com")

False
bbde8f7d


WebFetchResponse(title='Example Domain', content='Example Domain\n\n# Example Domain\n\nThis domain is for use in documentation examples without needing permission. Avoid use in operations.\n\nExample Domain\n# Example Domain\nThis domain is for use in documentation examples without needing permission. Avoid use in operations.\n[Learn more](https://iana.org/domains/example)', links=['https://iana.org/domains/example'])

In [28]:
for i in range(3):
    farm_name = farms.loc[i, "name"]
    website = farms.loc[i, "Website"] if pd.notna(farms.loc[i, "Website"]) else ""

    result = research_farm(farm_name, website)

    print("------")
    print(farm_name)
    print(result)

------
Elmwood Farm
Here is the response:

{
    "website_found": true,
    "social_media_found": false,
    "sells_produce": true,
    "sells_only_flowers_or_honey": false,
    "evidence": [
        "Elmwood Farm has a website: https://www.elmwoodfarm.co/",
        "Elmwood Farm sells produce through various means, including CSA and farm stand.",
        "There is evidence of food sales on the donation/cart page, as indicated by the 'Become a Member Today!' section."
    ],
    "classification": "Urban farm with produce sales"
}
------
Joppy Momma's Farm
{"name": "answer", "parameters": {"url": "https://joppymommas.org/"}}
------
Hatcher Station Training Farm
The final answer to the user's query is not explicitly stated in the provided WebSearchResult snippets. However, based on the content of the snippets, it appears that Restorative Farms is a nonprofit organization that aims to promote sustainable urban farming practices and provide training and resources for local farmers in South